# 13.2 同一提問的兩份回答，如何對齊成資料？


一組偏好資料先固定提問「1+1=?」，較佳回答2，較差回答3。保存為三個欄位：**prompt**是共同前文，**chosen**是被選為較佳的續寫，**rejected**是另一份續寫。這些名稱記錄人給的標籤，不是模型自己發現的真值。

模型算每篇答案的機率時，要看那篇答案自己的前文。下面將兩條對話分別轉成輸入與下一位置目標。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
from tiny_perceptron.data import render_chat

pair = {"prompt": "1+1=?", "chosen": "2", "rejected": "3"}
for key in ["chosen", "rejected"]:
    x, y = render_chat(
        [
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair[key]},
        ]
    )
    print(key, "輸入", x.tolist(), "目標", y.tolist())

chosen 輸入 [1, 3, 57, 51, 57, 69, 71, 2, 4, 58] 目標 [-100, -100, -100, -100, -100, -100, -100, -100, 58, 2]
rejected 輸入 [1, 3, 57, 51, 57, 69, 71, 2, 4, 59] 目標 [-100, -100, -100, -100, -100, -100, -100, -100, 59, 2]


本專案預設按UTF-8位元組編碼，普通編號加8，另保留1作開頭、3作使用者、4作助手、2作結束。數字2與3因此是58與59。chosen輸入為`[1,3,57,51,57,69,71,2,4,58]`，rejected只將最後一項換59。

兩份目標前八項都是`-100`，表示不計這些前文位置；最後兩項是數字答案58或59，再接結束2。輸入索引8的助手標記4要預測數字；輸入最後的數字要預測結束。工具已完成下一位置對齊，後面不能再平移一次。

因此也不能把chosen的輸入搭rejected的標籤：算回答3之後結束的機率時，模型應已看見3，不是2。同題兩條續寫可以不同長，但各自的輸入、答案與結束都要對齊。

練習只將rejected改4，預測最後普通編號變60、chosen不變，再執行核對。若rejected連提問都換了，分數差也可能來自題目難度，已不是本次同題比較。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

自然資料也需要這樣核對。我們另外取UltraFeedback偏好資料的一小包100筆，按提問家族自行切成80／10／10，並不是官方測試集；提問與兩篇回答各截到最多120個UTF-8 byte，保留完整字元。這樣能讓小模型讀得下，卻可能刪掉原評分者看過的關鍵理由。來源對完整回答的chosen標籤，不保證截短後仍適用。因此這個支線只檢查資料與訓練通路，不用它替加法實驗證明自然聊天品質。

</details>
